# Leiden clustering for TNBC2
This notebook loads per-patient cell features from HDF5, runs PCA, builds a kNN graph,
clusters with Leiden, summarizes clusters, chooses exemplars, and saves results.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR

from clearit.leiden.io import inspect_hdf5, list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph
from clearit.leiden.cluster import leiden
from clearit.leiden.exemplars import select_exemplars

# Reproducibility
SEED = 13

# Paths for both datasets
H5_TNBC2 = EMBEDDINGS_DIR / "TNBC2-MIBI44" / "DeepCell_MC17"  / "01_features-expressions" / "tnbc2-mibi8.hdf5"  # confirm filename

OUT_DIR = OUTPUTS_DIR / "leiden"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Clustering parameters
PCA_DIMS   = 64
KNN_K      = 85
LEI_RES    = 1.3

In [3]:
DATASET = "TNBC2"
h5_path = H5_TNBC2
assert h5_path.exists(), f"Missing file: {h5_path}"

# Inspect structure (optional)
summary = inspect_hdf5(h5_path)
print(f"Found {len(summary['groups'])} groups and {len(summary['datasets'])} datasets.")

Found 40 groups and 120 datasets.


In [6]:
# Discover patients and define a train/test split.
available = list_patients(h5_path)
print(f"{DATASET}: {len(available)} patient groups.")

# Here we use first 30 as train and the rest as test for TNBC2-like counts.
TRAIN_PTS = [p for p in available if int(p[1:]) <= 30]
TEST_PTS  = [p for p in available if p not in TRAIN_PTS]

SPLIT = "train"  # "train" or "test"
patients_to_load = TRAIN_PTS if SPLIT == "train" else TEST_PTS
print(f"Using {SPLIT} split with {len(patients_to_load)} patients.")

TNBC2: 40 patient groups.
Using train split with 29 patients.


In [7]:
# Load data with optional caps for large datasets.
X, meta, E, y = load_hdf5_split(
    h5_path=h5_path,
    patients=patients_to_load,
    load_expressions=False,
    load_labels=False,
    per_patient_cap=10_000,
    global_cap=150_000,
    seed=SEED,
)
# Extract only features corresponding to TNBC2-MIBI8 channels
n_channels = 44
n_features = 32

X_reshaped = X.reshape(X.shape[0], n_channels, n_features)

# desired channels (1-indexed → convert to 0-based)
channels = [1, 8, 10, 15, 17, 18, 19, 36] # Background, CD20, CD3, CD56, CD68, CD8, dsDNA, Pan-Keratin

# extract only those channels
X_selected = X_reshaped[:, channels, :]

# flatten the channel and feature dimensions back together
X = X_selected.reshape(X.shape[0], -1)
print(f"Loaded features: X shape = {X.shape}, meta rows = {len(meta)}")
display(meta.head())

Loaded features: X shape = (150000, 256), meta rows = 150000


,patient,idx_within_patient
0,P01,0
1,P01,1
2,P01,2
3,P01,3
4,P01,4


In [8]:
# Standardize and PCA
X_pca, scaler, pca = standardize_and_pca(X, n_components=PCA_DIMS, seed=SEED)
print(f"PCA: first 10 EVR = {np.round(pca.explained_variance_ratio_[:10], 4)}")
print(f"PCA: cumulative (k={PCA_DIMS}) = {pca.explained_variance_ratio_[:PCA_DIMS].sum():.4f}")

PCA: first 10 EVR = [0.1535 0.0637 0.0581 0.0478 0.0464 0.0399 0.0372 0.0293 0.0268 0.0252]
PCA: cumulative (k=64) = 0.9501


In [9]:
# Build kNN graph and run Leiden
g = build_knn_graph(X_pca, k=KNN_K, metric="euclidean")
labels, n_clusters = leiden(g, resolution=LEI_RES, seed=SEED)

meta = meta.copy()
meta["cluster_id"] = labels
print(f"Leiden: {n_clusters} clusters")

Leiden: 15 clusters


In [10]:
# Quick cluster summary
cluster_sizes = meta["cluster_id"].value_counts().sort_index()
summary = (
    meta.groupby("cluster_id")["patient"]
    .nunique()
    .rename("n_patients")
    .to_frame()
    .assign(size=cluster_sizes.values)
    .reset_index()
    .sort_values("size", ascending=False)
)
print(f"Total clusters: {summary.shape[0]}")
display(summary)

Total clusters: 15


,cluster_id,n_patients,size
0,0,26,20666
1,1,27,18033
2,2,29,16586
3,3,29,15379
4,4,29,13295
5,5,28,10364
6,6,27,9646
7,7,29,8943
8,8,29,8888
9,9,24,7146


In [11]:
# Exemplar selection
EXEMPLARS_PER_CLUSTER = 10
EXEMPLARS_PER_PATIENT_MAX = 2
DENSITY_K = 15

exemplars = select_exemplars(
    X_pca=X_pca,
    meta=meta,
    labels=labels,
    exemplars_per_cluster=EXEMPLARS_PER_CLUSTER,
    per_patient_max=EXEMPLARS_PER_PATIENT_MAX,
    density_k=DENSITY_K,
)
print(
    f"Selected {len(exemplars)} exemplars across "
    f"{exemplars['cluster_id'].nunique()} clusters."
)
display(exemplars.head(20))

Selected 150 exemplars across 15 clusters.


,patient,idx_within_patient,cluster_id,exemplar_rank
27907,P06,1348,0,1.0
18392,P04,3882,0,2.0
44256,P09,5153,0,3.0
66793,P13,4864,0,4.0
16953,P04,2443,0,5.0
8632,P03,437,0,6.0
85267,P16,6088,0,7.0
27265,P06,706,0,9.0
42851,P09,3748,0,12.0
66263,P13,4334,0,18.0


In [12]:
# Save outputs
assign_csv    = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_clusters.csv"
exemplars_csv = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_exemplars.csv"

meta.to_csv(assign_csv, index=False)
exemplars.to_csv(exemplars_csv, index=False)

print("Saved:")
print(" -", assign_csv)
print(" -", exemplars_csv)


Saved:
 - /workspace/files/CLEAR-IT/outputs/leiden/tnbc2_train_clusters.csv
 - /workspace/files/CLEAR-IT/outputs/leiden/tnbc2_train_exemplars.csv


In [1]:
#!/usr/bin/env python3
"""
Scan kNN-k and Leiden resolution; report cluster count, stability (mean ARI), and quality.

Outputs:
  - CSV with all runs: <OUTPUTS_DIR>/leiden/<dataset>_<split>_scan.csv
  - Console summary of settings closest to target cluster count (default: 14)

Dependencies:
  clearit.leiden.{io, preprocess, graph}
  numpy, pandas, scikit-learn, igraph, leidenalg
"""

from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple
import itertools
import numpy as np
import pandas as pd

import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_rand_score

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR
from clearit.leiden.io import list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph


# ----------------------------- Configuration -----------------------------

# Dataset selection
DATASET = "TNBC2"
SPLIT = "train"    # "train" or "test"

# HDF5 paths
H5_TNBC2 = EMBEDDINGS_DIR / "TNBC2-MIBI44" / "DeepCell_MC17" / "01_features-expressions" / "tnbc2-mibi8.hdf5"

# Embedding + graph defaults (centered on your current working point)
PCA_DIMS_DEFAULT = 64
K_GRID  = [55, 65, 70, 75, 85]
RES_GRID = [1.0, 1.2, 1.3, 1.4, 1.6]
SEEDS = [0, 1, 2, 3, 4]
TARGET_CLUSTERS = 14

# Subsampling caps (set to None for full data)
PER_PATIENT_CAP = 10_000
GLOBAL_CAP = 150_000

# Reproducibility
SEED = 42

# Output
OUT_DIR = OUTPUTS_DIR / "leiden"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------- Utilities -------------------------------

@dataclass
class PartitionStats:
    labels: np.ndarray
    quality: float
    n_clusters: int


def run_leiden_partition(
    g: ig.Graph,
    resolution: float,
    seed: int,
) -> PartitionStats:
    """
    Run Leiden with RBConfigurationVertexPartition; return labels, quality, and cluster count.
    """
    part = la.find_partition(
        g,
        la.RBConfigurationVertexPartition,
        weights=g.es["weight"] if "weight" in g.es.attributes() else None,
        resolution_parameter=resolution,
        seed=seed,
    )
    labels = np.asarray(part.membership, dtype=int)
    quality = float(part.quality())  # objective value for the chosen partition type
    n_clusters = int(labels.max() + 1) if labels.size else 0
    return PartitionStats(labels=labels, quality=quality, n_clusters=n_clusters)


def mean_pairwise_ari(labels_list: List[np.ndarray]) -> float:
    """
    Compute mean Adjusted Rand Index over all unique pairs in a list of labelings.
    """
    if len(labels_list) < 2:
        return 1.0
    pairs = [(i, j) for i in range(len(labels_list)) for j in range(i + 1, len(labels_list))]
    aris = [adjusted_rand_score(labels_list[i], labels_list[j]) for i, j in pairs]
    return float(np.mean(aris))


def choose_split_patients(h5_path: Path) -> Tuple[list[str], list[str]]:
    """
    Build a simple train/test split over available patient groups.
    For TNBC2-like counts, the first 30 patients are train; remainder are test.
    For other counts, split 75/25 by index.
    """
    pts = list_patients(h5_path)
    if len(pts) >= 41:
        train = [p for p in pts if int(p[1:]) <= 30]
        test = [p for p in pts if p not in train]
        return train, test
    # Generic fallback split
    n_train = max(1, int(0.75 * len(pts)))
    return pts[:n_train], pts[n_train:]


# --------------------------------- Main ----------------------------------

def main():
    # Resolve dataset path
    h5_path = H5_TNBC2
    assert h5_path.exists(), f"Missing file: {h5_path}"

    # Determine patients for chosen split
    train_pts, test_pts = choose_split_patients(h5_path)
    patients = train_pts if SPLIT == "train" else test_pts
    print(f"{DATASET} | {SPLIT}: {len(patients)} patients")

    # Load and embed once; graph is rebuilt per k
    X, meta, _, _ = load_hdf5_split(
        h5_path=h5_path,
        patients=patients,
        load_expressions=False,
        load_labels=False,
        per_patient_cap=PER_PATIENT_CAP,
        global_cap=GLOBAL_CAP,
        seed=SEED,
    )
    # Extract only features corresponding to TNBC2-MIBI8 channels
    n_channels = 44
    n_features = 32

    X_reshaped = X.reshape(X.shape[0], n_channels, n_features)

    # desired channels (1-indexed → convert to 0-based)
    channels = [1, 8, 10, 15, 17, 18, 19, 36] # Background, CD20, CD3, CD56, CD68, CD8, dsDNA, Pan-Keratin

    # extract only those channels
    X_selected = X_reshaped[:, channels, :]

    # flatten the channel and feature dimensions back together
    X = X_selected.reshape(X.shape[0], -1)
    print(f"Features loaded: X = {X.shape}, meta rows = {len(meta)}")

    X_pca, scaler, pca = standardize_and_pca(X, n_components=PCA_DIMS_DEFAULT, seed=SEED)
    cum_var = float(pca.explained_variance_ratio_[:PCA_DIMS_DEFAULT].sum())
    print(f"PCA dims = {PCA_DIMS_DEFAULT}, cumulative explained variance = {cum_var:.4f}")

    # Scan grid
    rows = []
    for k, res in itertools.product(K_GRID, RES_GRID):
        # Build kNN graph for this k
        g = build_knn_graph(X_pca, k=k, metric="euclidean")

        # Precompute some graph stats for reference
        n = g.vcount()
        m = g.ecount()
        mean_deg = float(np.mean(g.degree())) if n > 0 else 0.0
        n_components = len(g.components())

        # Multiple seeds for stability
        label_runs = []
        qualities = []
        ncls = []
        for s in SEEDS:
            stats = run_leiden_partition(g, resolution=res, seed=s)
            label_runs.append(stats.labels)
            qualities.append(stats.quality)
            ncls.append(stats.n_clusters)

        # Stability and summary metrics
        mean_ari = mean_pairwise_ari(label_runs)
        mean_clusters = float(np.mean(ncls))
        std_clusters = float(np.std(ncls))
        mean_quality = float(np.mean(qualities))
        std_quality = float(np.std(qualities))

        rows.append(
            dict(
                dataset=DATASET,
                split=SPLIT,
                pca_dims=PCA_DIMS_DEFAULT,
                k=k,
                resolution=res,
                seeds=len(SEEDS),
                mean_clusters=mean_clusters,
                std_clusters=std_clusters,
                mean_ari=mean_ari,
                mean_quality=mean_quality,
                std_quality=std_quality,
                n_vertices=n,
                n_edges=m,
                mean_degree=mean_deg,
                n_components=n_components,
                target=TARGET_CLUSTERS,
                abs_diff_from_target=abs(mean_clusters - TARGET_CLUSTERS),
            )
        )
        print(
            f"k={k:>2}, res={res:>3.1f} -> "
            f"clusters {mean_clusters:.1f}±{std_clusters:.1f}, "
            f"ARI={mean_ari:.3f}, quality={mean_quality:.4f}, "
            f"|Δ|={abs(mean_clusters - TARGET_CLUSTERS):.1f}"
        )

    df = pd.DataFrame(rows).sort_values(
        ["abs_diff_from_target", "mean_ari", "mean_quality"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    # Save results
    out_csv = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_scan.csv"
    df.to_csv(out_csv, index=False)
    print(f"\nSaved scan results: {out_csv}")

    # Console summary: top candidates near target cluster count
    top = df.head(10)[
        [
            "k",
            "resolution",
            "mean_clusters",
            "std_clusters",
            "mean_ari",
            "mean_quality",
            "mean_degree",
            "n_components",
        ]
    ]
    print("\nTop candidates near target cluster count:")
    print(top.to_string(index=False))


if __name__ == "__main__":
    main()


TNBC2 | train: 30 patients
Features loaded: X = (150000, 256), meta rows = 150000
PCA dims = 64, cumulative explained variance = 0.9501
k=55, res=1.0 -> clusters 12.6±0.5, ARI=0.895, quality=914489.3014, |Δ|=1.4
k=55, res=1.2 -> clusters 14.2±0.4, ARI=0.890, quality=889998.0270, |Δ|=0.2
k=55, res=1.3 -> clusters 14.8±0.7, ARI=0.884, quality=878367.4825, |Δ|=0.8
k=55, res=1.4 -> clusters 15.0±0.6, ARI=0.869, quality=866869.8412, |Δ|=1.0
k=55, res=1.6 -> clusters 17.0±0.6, ARI=0.788, quality=844467.4706, |Δ|=3.0
k=65, res=1.0 -> clusters 12.4±0.5, ARI=0.889, quality=1064302.8100, |Δ|=1.6
k=65, res=1.2 -> clusters 13.8±0.4, ARI=0.887, quality=1034886.2684, |Δ|=0.2
k=65, res=1.3 -> clusters 14.4±0.5, ARI=0.903, quality=1021430.2561, |Δ|=0.4
k=65, res=1.4 -> clusters 14.8±0.7, ARI=0.882, quality=1007909.7444, |Δ|=0.8
k=65, res=1.6 -> clusters 17.4±0.5, ARI=0.753, quality=980537.7720, |Δ|=3.4
k=70, res=1.0 -> clusters 12.6±0.8, ARI=0.842, quality=1137111.5792, |Δ|=1.4
k=70, res=1.2 -> cluste

In [ ]:
#!/usr/bin/env python3
"""
Parallel scan of kNN-k and Leiden resolution with stability (mean ARI) and quality.

This version:
- Runs each (k, resolution) combo in a separate process (N_JOBS workers).
- Limits BLAS threads per worker to avoid oversubscription.
- Optionally uses multiple threads inside scikit-learn's NearestNeighbors if available.

Outputs:
  <OUTPUTS_DIR>/leiden/<dataset>_<split>_scan.csv
"""

from __future__ import annotations
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any
import itertools
import numpy as np
import pandas as pd
import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_rand_score
from concurrent.futures import ProcessPoolExecutor, as_completed

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR
from clearit.leiden.io import list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph

# ----------------------------- Configuration -----------------------------

DATASET = "TNBC2"     # "TNBC1" or "TNBC2"
SPLIT   = "train"     # "train" or "test"

# Embedding + graph defaults (centered on your current working point)
PCA_DIMS_DEFAULT = 64
K_GRID  = [55, 65, 70, 75, 85]
RES_GRID = [1.0, 1.2, 1.3, 1.4, 1.6]
SEEDS = [0, 1, 2, 3, 4]
TARGET_CLUSTERS = 14

# Subsampling caps
PER_PATIENT_CAP = 10_000
GLOBAL_CAP      = 150_000

# Parallelism
N_JOBS = os.cpu_count() or 4           # number of worker processes
NN_N_JOBS = 1                          # suggested per-worker threads for NearestNeighbors (see note below)

# Reproducibility
SEED = 42

# HDF5 paths
H5_TNBC1 = EMBEDDINGS_DIR / "TNBC1-MxIF8"  / "inForm_MC7"    / "01_features-expressions" / "tnbc1-mxif8.hdf5"
H5_TNBC2 = EMBEDDINGS_DIR / "TNBC2-MIBI44" / "DeepCell_MC17" / "01_features-expressions" / "tnbc2-mibi8.hdf5"

OUT_DIR = OUTPUTS_DIR / "leiden"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------- Utilities -------------------------------

@dataclass
class PartitionStats:
    labels: np.ndarray
    quality: float
    n_clusters: int

def run_leiden_partition(g: ig.Graph, resolution: float, seed: int) -> PartitionStats:
    """
    Run Leiden with RBConfigurationVertexPartition; return labels, quality, and cluster count.
    """
    part = la.find_partition(
        g,
        la.RBConfigurationVertexPartition,
        weights=g.es["weight"] if "weight" in g.es.attributes() else None,
        resolution_parameter=resolution,
        seed=seed,
    )
    labels = np.asarray(part.membership, dtype=int)
    quality = float(part.quality())
    n_clusters = int(labels.max() + 1) if labels.size else 0
    return PartitionStats(labels=labels, quality=quality, n_clusters=n_clusters)

def mean_pairwise_ari(labels_list: List[np.ndarray]) -> float:
    """
    Compute mean Adjusted Rand Index over all unique label-pairs.
    """
    if len(labels_list) < 2:
        return 1.0
    pairs = [(i, j) for i in range(len(labels_list)) for j in range(i + 1, len(labels_list))]
    aris = [adjusted_rand_score(labels_list[i], labels_list[j]) for i, j in pairs]
    return float(np.mean(aris))

def choose_split_patients(h5_path: Path) -> Tuple[list[str], list[str]]:
    """
    Build a train/test split over available patient groups. For TNBC1-like counts,
    first 47 as train; remainder as test. Otherwise use a 75/25 split by index.
    """
    pts = list_patients(h5_path)
    if len(pts) >= 63:
        train = [p for p in pts if int(p[1:]) <= 47]
        test = [p for p in pts if p not in train]
        return train, test
    n_train = max(1, int(0.75 * len(pts)))
    return pts[:n_train], pts[n_train:]

# --------------------------- Worker entry point ---------------------------

def _configure_worker_blas_threads():
    """
    Limit BLAS threads to avoid oversubscription when using process parallelism.
    """
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

def _build_graph_with_optional_threads(X_pca: np.ndarray, k: int, nn_n_jobs: int) -> ig.Graph:
    """
    Build the kNN graph. If the installed scikit-learn supports n_jobs in NearestNeighbors,
    use it; otherwise fall back to default behavior.
    """
    # Use the project helper; if you prefer to pass n_jobs down, adapt your build_knn_graph.
    # Below is a minimal compatibility shim in case your sklearn exposes n_jobs.
    try:
        from sklearn.neighbors import NearestNeighbors
        # Test if NearestNeighbors supports n_jobs in this environment.
        _ = NearestNeighbors(n_neighbors=2, n_jobs=nn_n_jobs)
        # If no error, rebuild here with threads; then mirror the logic from clearit.leiden.graph
        nn = NearestNeighbors(n_neighbors=k, metric="euclidean", n_jobs=nn_n_jobs)
        nn.fit(X_pca)
        dists, nbrs = nn.kneighbors(X_pca, return_distance=True)
        import pandas as pd
        N = X_pca.shape[0]
        rows = np.repeat(np.arange(N), k)
        cols = nbrs.ravel()
        weights = 1.0 / (1.0 + dists.ravel())
        mask = rows != cols
        rows, cols, weights = rows[mask], cols[mask], weights[mask]
        u = np.minimum(rows, cols)
        v = np.maximum(rows, cols)
        edges_df = pd.DataFrame({"u": u, "v": v, "w": weights})
        edges_df = edges_df.sort_values(["u", "v", "w"], ascending=[True, True, False]).drop_duplicates(["u", "v"], keep="first")
        g = ig.Graph(n=N, edges=list(zip(edges_df.u.values, edges_df.v.values)), directed=False)
        g.es["weight"] = edges_df.w.values
        return g
    except Exception:
        # Fallback to the project helper (which may be compiled with its own threading)
        return build_knn_graph(X_pca, k=k, metric="euclidean")

def evaluate_combo(args: Tuple[int, float, np.ndarray, List[int], int]) -> Dict[str, Any]:
    """
    Worker that evaluates one (k, resolution) pair across multiple seeds.
    Returns a single summary row as a dict.
    """
    k, res, X_pca, seeds, target = args
    _configure_worker_blas_threads()

    g = _build_graph_with_optional_threads(X_pca, k=k, nn_n_jobs=NN_N_JOBS)

    n = g.vcount()
    m = g.ecount()
    mean_deg = float(np.mean(g.degree())) if n > 0 else 0.0
    n_components = len(g.components())

    label_runs, qualities, ncls = [], [], []
    for s in seeds:
        stats = run_leiden_partition(g, resolution=res, seed=s)
        label_runs.append(stats.labels)
        qualities.append(stats.quality)
        ncls.append(stats.n_clusters)

    mean_ari = mean_pairwise_ari(label_runs)
    mean_clusters = float(np.mean(ncls))
    std_clusters = float(np.std(ncls))
    mean_quality = float(np.mean(qualities))
    std_quality = float(np.std(qualities))

    return dict(
        k=k,
        resolution=res,
        seeds=len(seeds),
        mean_clusters=mean_clusters,
        std_clusters=std_clusters,
        mean_ari=mean_ari,
        mean_quality=mean_quality,
        std_quality=std_quality,
        n_vertices=n,
        n_edges=m,
        mean_degree=mean_deg,
        n_components=n_components,
        target=target,
        abs_diff_from_target=abs(mean_clusters - target),
    )

# --------------------------------- Main ----------------------------------

def main():
    h5_path = H5_TNBC1 if DATASET.upper() == "TNBC1" else H5_TNBC2
    assert h5_path.exists(), f"Missing file: {h5_path}"

    train_pts, test_pts = choose_split_patients(h5_path)
    patients = train_pts if SPLIT == "train" else test_pts
    print(f"{DATASET} | {SPLIT}: {len(patients)} patients")

    X, meta, _, _ = load_hdf5_split(
        h5_path=h5_path,
        patients=patients,
        load_expressions=False,
        load_labels=False,
        per_patient_cap=PER_PATIENT_CAP,
        global_cap=GLOBAL_CAP,
        seed=SEED,
    )
    # Extract only features corresponding to TNBC2-MIBI8 channels
    n_channels = 44
    n_features = 32

    X_reshaped = X.reshape(X.shape[0], n_channels, n_features)

    # desired channels (1-indexed → convert to 0-based)
    channels = [1, 8, 10, 15, 17, 18, 19, 36] # Background, CD20, CD3, CD56, CD68, CD8, dsDNA, Pan-Keratin

    # extract only those channels
    X_selected = X_reshaped[:, channels, :]

    # flatten the channel and feature dimensions back together
    X = X_selected.reshape(X.shape[0], -1)
    print(f"Features loaded: X = {X.shape}, meta rows = {len(meta)}")

    X_pca, scaler, pca = standardize_and_pca(X, n_components=PCA_DIMS_DEFAULT, seed=SEED)
    cum_var = float(pca.explained_variance_ratio_[:PCA_DIMS_DEFAULT].sum())
    print(f"PCA dims = {PCA_DIMS_DEFAULT}, cumulative explained variance = {cum_var:.4f}")

    combos = list(itertools.product(K_GRID, RES_GRID))
    print(f"Evaluating {len(combos)} (k, resolution) combos with {len(SEEDS)} seeds each using {N_JOBS} workers...")

    results: List[Dict[str, Any]] = []
    with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
        futures = {
            ex.submit(evaluate_combo, (k, res, X_pca, SEEDS, TARGET_CLUSTERS)): (k, res)
            for (k, res) in combos
        }
        for fut in as_completed(futures):
            k, res = futures[fut]
            row = fut.result()
            results.append(row)
            print(
                f"k={k:>3}, res={res:>3.2f} -> "
                f"clusters {row['mean_clusters']:.1f}±{row['std_clusters']:.1f}, "
                f"ARI={row['mean_ari']:.3f}, quality={row['mean_quality']:.4f}, "
                f"|Δ|={row['abs_diff_from_target']:.1f}"
            )

    df = pd.DataFrame(results).sort_values(
        ["abs_diff_from_target", "mean_ari", "mean_quality"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    out_csv = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_scan.csv"
    df.to_csv(out_csv, index=False)
    print(f"\nSaved scan results: {out_csv}")

    top = df.head(10)[
        [
            "k", "resolution", "mean_clusters", "std_clusters",
            "mean_ari", "mean_quality", "mean_degree", "n_components",
        ]
    ]
    print("\nTop candidates near target cluster count:")
    print(top.to_string(index=False))


if __name__ == "__main__":
    main()


TNBC2 | train: 30 patients
Features loaded: X = (150000, 256), meta rows = 150000
PCA dims = 64, cumulative explained variance = 0.9501
Evaluating 25 (k, resolution) combos with 5 seeds each using 20 workers...
